ARTI308 - Machine Learning

# Lab 6 – Task: Linear Regression on Ecommerce Customers Dataset

In this task, we apply Linear Regression to predict how much money a customer spends yearly, based on their behavior on a company's app and website.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics

%matplotlib inline
sns.set(style='whitegrid')

## 2. Load the Dataset

In [ ]:
df = pd.read_csv('Ecommerce_Customers')
df.head()

## 3. Explore the Data

In [ ]:
df.info()

In [ ]:
df.describe()

The dataset contains **500 rows** and **8 columns**:
- `Email`, `Address`, `Avatar` — text columns, not useful for modeling
- `Avg. Session Length`, `Time on App`, `Time on Website`, `Length of Membership` — numerical features
- `Yearly Amount Spent` — **target variable** (what we want to predict)

## 4. Data Cleaning

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Duplicate rows:', df.duplicated().sum())

The dataset is already clean — no missing values and no duplicate rows. No cleaning is needed.

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Pairplot to visualize relationships between numerical features and the target
sns.pairplot(df[['Avg. Session Length', 'Time on App', 'Time on Website',
                  'Length of Membership', 'Yearly Amount Spent']])
plt.suptitle('Pairplot of Numerical Features', y=1.02)
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(8, 5))
sns.heatmap(df[['Avg. Session Length', 'Time on App', 'Time on Website',
                 'Length of Membership', 'Yearly Amount Spent']].corr(),
            annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

**Key observations:**
- `Length of Membership` has the strongest correlation with `Yearly Amount Spent`
- `Time on App` also shows a strong positive relationship
- `Time on Website` has very little correlation with spending

## 6. Feature Engineering

No feature engineering is needed here. The numerical columns are already meaningful and ready to use. We only drop the text columns (`Email`, `Address`, `Avatar`) since linear regression cannot use them.

## 7. Prepare the Data for Modeling

We select the numerical features as **X** and `Yearly Amount Spent` as **y**.

In [ ]:
X = df[['Avg. Session Length', 'Time on App', 'Time on Website', 'Length of Membership']]
y = df['Yearly Amount Spent']

print('X shape:', X.shape)
print('y shape:', y.shape)

## 8. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=101
)

print('Train size:', X_train.shape)
print('Test size:', X_test.shape)

We use **70% for training** and **30% for testing**.

## 9. Train the Linear Regression Model

In [ ]:
lm = LinearRegression()
lm.fit(X_train, y_train)

print('Model trained successfully!')

### Coefficients

In [ ]:
coeff_df = pd.DataFrame(lm.coef_, X.columns, columns=['Coefficient'])
print('Intercept:', lm.intercept_)
print()
coeff_df

**Interpreting the coefficients** (holding all other features fixed):

- A 1-unit increase in **Avg. Session Length** is associated with an increase of **\$25.98** in yearly spending.
- A 1-unit increase in **Time on App** is associated with an increase of **\$38.59** in yearly spending.
- A 1-unit increase in **Time on Website** is associated with an increase of **\$0.19** in yearly spending — almost no effect.
- A 1-unit increase in **Length of Membership** is associated with an increase of **\$61.28** in yearly spending — the strongest predictor.

## 10. Predictions

In [ ]:
predictions = lm.predict(X_test)

# Scatter plot: Actual vs Predicted
plt.figure(figsize=(6, 4))
plt.scatter(y_test, predictions, alpha=0.6)
plt.xlabel('Actual Yearly Amount Spent')
plt.ylabel('Predicted Yearly Amount Spent')
plt.title('Actual vs Predicted')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')  # perfect prediction line
plt.show()

The points fall very close to the red diagonal line, which means the model predictions are close to the actual values — a good sign.

In [ ]:
# Residuals histogram
residuals = y_test - predictions

plt.figure(figsize=(6, 4))
sns.histplot(residuals, bins=40, kde=True)
plt.xlabel('Residuals')
plt.title('Residuals Distribution')
plt.show()

The residuals are approximately normally distributed and centered around 0 — this confirms the linear regression assumptions are well satisfied.

## 11. Model Evaluation

In [ ]:
mae  = metrics.mean_absolute_error(y_test, predictions)
mse  = metrics.mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2   = metrics.r2_score(y_test, predictions)

print(f'MAE  (Mean Absolute Error):       {mae:.4f}')
print(f'MSE  (Mean Squared Error):        {mse:.4f}')
print(f'RMSE (Root Mean Squared Error):   {rmse:.4f}')
print(f'R²   (R-squared):                 {r2:.4f}')

**Evaluation Summary:**

| Metric | Value |
|--------|-------|
| MAE    | ~7.23 |
| MSE    | ~79.81 |
| RMSE   | ~8.93 |
| R²     | ~0.989 |

- **MAE of ~7.23** means the model is off by about \$7.23 on average — very low.
- **RMSE of ~8.93** is also very small relative to the target range (\$256 – \$765).
- **R² of 0.989** means the model explains **98.9%** of the variance in yearly spending — an excellent fit.

**Conclusion:** The Linear Regression model performs very well on this dataset. `Length of Membership` and `Time on App` are the most important predictors of how much a customer spends yearly.